# DACT-EXP-020B — Driver Action Classifier Baseline

Amaç: `DRIVER-EXP-001` driver presence gate tamamlandıktan sonra ilk sürücü eylemi sınıflandırıcısını kurmak.

Bu notebook ilk aşamada **State Farm Distracted Driver Detection** verisini kullanır ve FTR `sofor_eylemi` için şu internal etiketleri üretir:

- `telefonla_konusma`
- `su_icme`
- `arkaya_bakma_candidate`
- `phone_use_non_call`
- `passenger_interaction_candidate`
- `other_distraction_hard_negative`
- `safe_or_no_event`

Önemli sınır: Bu model tek başına hukuki/ihlal kararı vermez. `telefonla_konusma` ve `su_icme` için ilk classifier baseline'dır; `arkaya_bakma_candidate` weak label olarak kalır ve head/torso gate ile güçlendirilmeden final `arkaya_bakma` sayılmaz.

Varsayılan mod `quick`: pipeline'ı hızlı doğrulamak için dengeli subset ve MobileNetV3 koşar. Daha ağır run için Cell 2'de `RUN_MODE = 'full'` yapılır.


## Senin Yapman Gerekenler

1. Colab runtime: önce **T4** yeterli. Full run için L4/A100 daha hızlıdır.
2. Drive mount izni ver.
3. State Farm zip şu yollardan biriyle hazır olmalı:
   - `/content/drive/MyDrive/anomali-road-safety-ai/datasets/cabin_exp_020a/state_farm/state-farm-distracted-driver-detection.zip`
   - veya chunk dosyaları: `.../state_farm/chunks/state-farm-distracted-driver-detection.zip.part-*`
   - veya Kaggle credentials ile resmi competition download.
4. Kaggle kullanılacaksa Colab Secrets içinde şu isimleri tanımla:
   - `KAGGLE_USERNAME`
   - `KAGGLE_KEY`

Not: Kaggle 401 verirse bu genellikle API key değil, competition/data erişimi veya hesap eşleşmesi problemidir. Zip'i Drive'a manuel koymak daha stabil yoldur.


In [ ]:
# Cell 1 — Runtime setup + dependencies
import os
import sys
import json
import math
import time
import random
import shutil
import zipfile
import subprocess
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone


def run_cmd(cmd, check=True, cwd=None):
    print('+', ' '.join(map(str, cmd)))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print('--- stdout ---')
        print(result.stdout[-4000:])
    if result.stderr:
        print('--- stderr ---')
        print(result.stderr[-4000:])
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with code {result.returncode}: {cmd}')
    return result

# Colab usually has most packages. Install only missing essentials.
try:
    import torch
    import torchvision
    import pandas as pd
    import numpy as np
    import sklearn
    import matplotlib
except Exception:
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'torchvision', 'pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'tqdm'])

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score, f1_score
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device count:', torch.cuda.device_count())


In [ ]:
# Cell 2 — Drive mount + configuration
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/anomali-road-safety-ai') if IN_COLAB else Path.cwd()
LOCAL_ROOT = Path('/content/anomali-road-safety-ai-work') if IN_COLAB else Path.cwd() / '.local_colab_work'

EXP_ID = 'DACT-EXP-020B'
EXP_NAME = 'statefarm_driver_action_classifier_v1'

# quick: pipeline smoke / first run. full: larger training.
RUN_MODE = 'quick'  # 'quick' or 'full'
BACKBONES = ['mobilenet_v3_large'] if RUN_MODE == 'quick' else ['mobilenet_v3_large', 'efficientnet_b0']
EPOCHS = 3 if RUN_MODE == 'quick' else 12
MAX_IMAGES_PER_INTERNAL_LABEL = 800 if RUN_MODE == 'quick' else None
BATCH_SIZE = 64 if RUN_MODE == 'quick' else 96
IMG_SIZE = 224
NUM_WORKERS = 2
USE_WEIGHTED_SAMPLER = True
FREEZE_BACKBONE_EPOCHS = 1 if RUN_MODE == 'quick' else 2
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4

# Dataset paths. We intentionally reuse the State Farm cache from CABIN-EXP-020A.
DRIVE_STATE_FARM_ROOT = PROJECT_ROOT / 'datasets' / 'cabin_exp_020a' / 'state_farm'
DRIVE_STATE_FARM_ZIP = DRIVE_STATE_FARM_ROOT / 'state-farm-distracted-driver-detection.zip'
DRIVE_STATE_FARM_CHUNKS = DRIVE_STATE_FARM_ROOT / 'chunks'
LOCAL_STATE_FARM_ROOT = LOCAL_ROOT / 'datasets' / 'driver_action_exp_020b' / 'state_farm'
LOCAL_ARCHIVE_ROOT = LOCAL_ROOT / 'archives' / 'driver_action_exp_020b'
LOCAL_STATE_FARM_ZIP = LOCAL_ARCHIVE_ROOT / DRIVE_STATE_FARM_ZIP.name

DRIVE_EXP_ROOT = PROJECT_ROOT / 'datasets' / 'driver_action_exp_020b'
DRIVE_META_DIR = DRIVE_EXP_ROOT / 'metadata'
DRIVE_RUN_ROOT = PROJECT_ROOT / 'runs' / 'driver_action' / EXP_ID
DRIVE_CKPT_ROOT = PROJECT_ROOT / 'models' / 'checkpoints' / 'cabin_driver' / EXP_ID
DRIVE_ARTIFACT_ROOT = PROJECT_ROOT / 'models' / 'benchmarks' / 'artifacts' / 'cabin_driver' / EXP_ID
DRIVE_REPORT_PATH = PROJECT_ROOT / 'testing' / 'reports' / 'dact_exp_020b_driver_action_classifier.md'

for path in [LOCAL_ROOT, LOCAL_STATE_FARM_ROOT, LOCAL_ARCHIVE_ROOT, DRIVE_META_DIR, DRIVE_RUN_ROOT, DRIVE_CKPT_ROOT, DRIVE_ARTIFACT_ROOT, DRIVE_REPORT_PATH.parent]:
    path.mkdir(parents=True, exist_ok=True)

STATE_FARM_COMPETITION = 'state-farm-distracted-driver-detection'
ENABLE_KAGGLE_DOWNLOAD = True
KAGGLE_DATASET_FALLBACKS = []  # Example: ['rightway11/state-farm-distracted-driver-detection'] if you trust/use a mirror.

STATE_FARM_TO_INTERNAL = {
    'c0': 'safe_or_no_event',
    'c1': 'phone_use_non_call',
    'c2': 'telefonla_konusma',
    'c3': 'phone_use_non_call',
    'c4': 'telefonla_konusma',
    'c5': 'other_distraction_hard_negative',
    'c6': 'su_icme',
    'c7': 'arkaya_bakma_candidate',
    'c8': 'other_distraction_hard_negative',
    'c9': 'passenger_interaction_candidate',
}
INTERNAL_LABELS = [
    'safe_or_no_event',
    'telefonla_konusma',
    'phone_use_non_call',
    'su_icme',
    'arkaya_bakma_candidate',
    'passenger_interaction_candidate',
    'other_distraction_hard_negative',
]
LABEL_TO_IDX = {label: idx for idx, label in enumerate(INTERNAL_LABELS)}
IDX_TO_LABEL = {idx: label for label, idx in LABEL_TO_IDX.items()}

print('RUN_MODE:', RUN_MODE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('LOCAL_ROOT:', LOCAL_ROOT)
print('DRIVE_STATE_FARM_ROOT:', DRIVE_STATE_FARM_ROOT)
print('LOCAL_STATE_FARM_ROOT:', LOCAL_STATE_FARM_ROOT)
print('BACKBONES:', BACKBONES, 'EPOCHS:', EPOCHS, 'BATCH_SIZE:', BATCH_SIZE)


In [ ]:
# Cell 3 — Kaggle auth, zip reconstruction, and local extraction helpers

def setup_kaggle_credentials():
    username = os.environ.get('KAGGLE_USERNAME') or os.environ.get('kaggle_username')
    key = os.environ.get('KAGGLE_KEY') or os.environ.get('kaggle_key')
    if IN_COLAB:
        try:
            from google.colab import userdata
            username = username or userdata.get('KAGGLE_USERNAME') or userdata.get('kaggle_username')
            key = key or userdata.get('KAGGLE_KEY') or userdata.get('kaggle_key')
        except Exception as exc:
            print('Could not read Colab Secrets:', exc)
    if not username or not key:
        print('Kaggle credentials not available. Existing Drive zip/chunks will still be used if present.')
        return None
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = kaggle_dir / 'kaggle.json'
    kaggle_json.write_text(json.dumps({'username': username, 'key': key}), encoding='utf-8')
    kaggle_json.chmod(0o600)
    print('Kaggle credentials ready:', kaggle_json)
    return kaggle_json


def count_images(root: Path) -> int:
    if not root.exists():
        return 0
    return sum(1 for p in root.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})


def reconstruct_zip_from_chunks():
    if DRIVE_STATE_FARM_ZIP.exists() and DRIVE_STATE_FARM_ZIP.stat().st_size > 1_000_000_000:
        print('State Farm zip already exists:', DRIVE_STATE_FARM_ZIP, 'size GB:', round(DRIVE_STATE_FARM_ZIP.stat().st_size / 1e9, 3))
        return True
    chunks = sorted(DRIVE_STATE_FARM_CHUNKS.glob('state-farm-distracted-driver-detection.zip.part-*'))
    if not chunks:
        print('No State Farm chunks found:', DRIVE_STATE_FARM_CHUNKS)
        return False
    print('Reconstructing State Farm zip from chunks:', len(chunks))
    DRIVE_STATE_FARM_ROOT.mkdir(parents=True, exist_ok=True)
    tmp = DRIVE_STATE_FARM_ZIP.with_suffix('.zip.reconstructing')
    with tmp.open('wb') as out:
        for chunk in tqdm(chunks, desc='Reconstructing zip'):
            with chunk.open('rb') as f:
                shutil.copyfileobj(f, out)
    tmp.replace(DRIVE_STATE_FARM_ZIP)
    print('Reconstructed:', DRIVE_STATE_FARM_ZIP, 'size GB:', round(DRIVE_STATE_FARM_ZIP.stat().st_size / 1e9, 3))
    return True


def kaggle_download_state_farm():
    if not ENABLE_KAGGLE_DOWNLOAD:
        return False
    if setup_kaggle_credentials() is None:
        return False
    DRIVE_STATE_FARM_ROOT.mkdir(parents=True, exist_ok=True)
    try:
        run_cmd(['kaggle', 'competitions', 'download', '-c', STATE_FARM_COMPETITION, '-p', str(DRIVE_STATE_FARM_ROOT)])
    except Exception as exc:
        print('Official competition download failed:', exc)
        print('If this is 401 Unauthorized, accept/join the Kaggle competition or put the zip in Drive manually.')
    if DRIVE_STATE_FARM_ZIP.exists():
        return True
    for slug in KAGGLE_DATASET_FALLBACKS:
        try:
            print('Trying fallback Kaggle dataset:', slug)
            run_cmd(['kaggle', 'datasets', 'download', '-d', slug, '-p', str(DRIVE_STATE_FARM_ROOT)])
            if DRIVE_STATE_FARM_ZIP.exists() or any(DRIVE_STATE_FARM_ROOT.glob('*.zip')):
                return True
        except Exception as exc:
            print('Fallback failed:', slug, exc)
    return DRIVE_STATE_FARM_ZIP.exists()


def find_state_farm_zip():
    reconstruct_zip_from_chunks()
    if not DRIVE_STATE_FARM_ZIP.exists():
        kaggle_download_state_farm()
    if DRIVE_STATE_FARM_ZIP.exists():
        return DRIVE_STATE_FARM_ZIP
    zips = sorted(DRIVE_STATE_FARM_ROOT.glob('*.zip'), key=lambda p: p.stat().st_size if p.exists() else 0, reverse=True)
    if zips:
        print('Using largest zip candidate:', zips[0])
        return zips[0]
    raise FileNotFoundError(
        'State Farm zip not found. Put state-farm-distracted-driver-detection.zip under:\n'
        f'  {DRIVE_STATE_FARM_ROOT}\n'
        'or provide chunks under chunks/ or fix Kaggle credentials/access.'
    )


def copy_zip_to_local(source_zip: Path) -> Path:
    """Avoid long ZipFile reads over Google Drive FUSE mount.

    Colab can drop `/content/drive` during thousands of small reads, producing
    `OSError: [Errno 107] Transport endpoint is not connected`. Copying the
    4.3 GB archive once to local runtime makes the later extraction local-only.
    """
    source_zip = Path(source_zip)
    if str(source_zip).startswith(str(LOCAL_ROOT)):
        return source_zip
    expected_size = source_zip.stat().st_size
    if LOCAL_STATE_FARM_ZIP.exists() and LOCAL_STATE_FARM_ZIP.stat().st_size == expected_size:
        print('Using existing local State Farm zip copy:', LOCAL_STATE_FARM_ZIP, 'size GB:', round(expected_size / 1e9, 3))
        return LOCAL_STATE_FARM_ZIP
    if LOCAL_STATE_FARM_ZIP.exists():
        print('Removing incomplete local zip copy:', LOCAL_STATE_FARM_ZIP, 'size:', LOCAL_STATE_FARM_ZIP.stat().st_size, 'expected:', expected_size)
        LOCAL_STATE_FARM_ZIP.unlink()
    tmp = LOCAL_STATE_FARM_ZIP.with_suffix('.zip.copying')
    if tmp.exists():
        tmp.unlink()
    print('Copying State Farm zip from Drive to local runtime before extraction:')
    print('  source:', source_zip)
    print('  target:', LOCAL_STATE_FARM_ZIP)
    try:
        with source_zip.open('rb') as src, tmp.open('wb') as dst:
            shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
        actual_size = tmp.stat().st_size
        if actual_size != expected_size:
            raise RuntimeError(f'Local zip copy size mismatch: got {actual_size}, expected {expected_size}')
        tmp.replace(LOCAL_STATE_FARM_ZIP)
    except OSError as exc:
        if tmp.exists():
            tmp.unlink()
        raise RuntimeError(
            'Google Drive mount disconnected while copying the State Farm zip. '
            'In Colab, run Runtime > Disconnect and delete runtime or remount Drive with force_remount=True, '
            'then rerun Cell 2 and Cell 3. Original error: ' + repr(exc)
        ) from exc
    print('Local State Farm zip copy ready:', LOCAL_STATE_FARM_ZIP, 'size GB:', round(LOCAL_STATE_FARM_ZIP.stat().st_size / 1e9, 3))
    return LOCAL_STATE_FARM_ZIP


def extract_member_subset(zip_path: Path, target_root: Path):
    target_root.mkdir(parents=True, exist_ok=True)
    marker = target_root / '.state_farm_train_extracted.marker'
    train_dir = target_root / 'imgs' / 'train'
    csv_path = target_root / 'driver_imgs_list.csv'
    if marker.exists() and train_dir.exists() and csv_path.exists() and count_images(train_dir) > 1000:
        print('State Farm local train already extracted:', train_dir, 'images:', count_images(train_dir))
        return
    print('Extracting State Farm train subset locally from:', zip_path)
    if train_dir.exists():
        shutil.rmtree(train_dir)
    if csv_path.exists():
        csv_path.unlink()
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        print('Zip member count:', len(names))
        direct_train = [n for n in names if n.startswith('imgs/train/') and not n.endswith('/')]
        driver_csv_names = [n for n in names if n.endswith('driver_imgs_list.csv')]
        nested_imgs = [n for n in names if n.endswith('imgs.zip')]
        if driver_csv_names:
            z.extract(driver_csv_names[0], target_root)
            extracted_csv = target_root / driver_csv_names[0]
            extracted_csv.parent.mkdir(parents=True, exist_ok=True)
            if extracted_csv != csv_path:
                shutil.move(str(extracted_csv), str(csv_path))
        if direct_train:
            print('Direct imgs/train members:', len(direct_train))
            for n in tqdm(direct_train, desc='Extract train images'):
                z.extract(n, target_root)
        elif nested_imgs:
            nested_name = nested_imgs[0]
            nested_path = target_root / 'imgs.zip'
            print('Nested imgs.zip detected:', nested_name)
            with z.open(nested_name) as src, nested_path.open('wb') as dst:
                shutil.copyfileobj(src, dst)
            with zipfile.ZipFile(nested_path) as nz:
                nnames = nz.namelist()
                direct_train = [n for n in nnames if n.startswith('imgs/train/') and not n.endswith('/')]
                print('Nested train members:', len(direct_train))
                for n in tqdm(direct_train, desc='Extract nested train images'):
                    nz.extract(n, target_root)
        else:
            raise RuntimeError('Zip contains neither imgs/train/* nor nested imgs.zip.')
    if not csv_path.exists():
        raise FileNotFoundError('driver_imgs_list.csv was not found/extracted.')
    image_count = count_images(train_dir)
    print('Extracted train image count:', image_count)
    if image_count < 1000:
        raise RuntimeError('Too few train images extracted; zip layout may be unsupported.')
    marker.write_text(datetime.now(timezone.utc).isoformat(), encoding='utf-8')

zip_path = copy_zip_to_local(find_state_farm_zip())
extract_member_subset(zip_path, LOCAL_STATE_FARM_ROOT)


In [ ]:
# Cell 4 — Build action metadata from State Farm labels
csv_path = LOCAL_STATE_FARM_ROOT / 'driver_imgs_list.csv'
train_root = LOCAL_STATE_FARM_ROOT / 'imgs' / 'train'
assert csv_path.exists(), csv_path
assert train_root.exists(), train_root

raw_df = pd.read_csv(csv_path)
required_cols = {'subject', 'classname', 'img'}
missing = required_cols - set(raw_df.columns)
if missing:
    raise ValueError('driver_imgs_list.csv missing columns: ' + ', '.join(sorted(missing)))

records = []
missing_images = 0
for row in raw_df.itertuples(index=False):
    source_class = str(row.classname)
    internal = STATE_FARM_TO_INTERNAL.get(source_class)
    if internal is None:
        continue
    image_path = train_root / source_class / str(row.img)
    if not image_path.exists():
        missing_images += 1
        continue
    records.append({
        'source': 'state_farm',
        'subject': str(row.subject),
        'source_class': source_class,
        'image_name': str(row.img),
        'image_path': str(image_path),
        'internal_label': internal,
        'target_idx': LABEL_TO_IDX[internal],
        'ftr_mapping': internal if internal in {'telefonla_konusma', 'su_icme'} else 'support_or_candidate',
    })

metadata_df = pd.DataFrame(records)
print('Raw records:', len(raw_df), 'usable:', len(metadata_df), 'missing_images:', missing_images)
print('Subjects:', metadata_df['subject'].nunique())
print('Source class counts:')
display(metadata_df['source_class'].value_counts().sort_index().rename('count').to_frame())
print('Internal label counts:')
display(metadata_df['internal_label'].value_counts().reindex(INTERNAL_LABELS).fillna(0).astype(int).rename('count').to_frame())

# Optional balanced quick subset after metadata creation.
if MAX_IMAGES_PER_INTERNAL_LABEL is not None:
    sampled_parts = []
    rng = np.random.default_rng(SEED)
    for label, group in metadata_df.groupby('internal_label'):
        n = min(len(group), MAX_IMAGES_PER_INTERNAL_LABEL)
        sampled_parts.append(group.sample(n=n, random_state=SEED))
    metadata_df = pd.concat(sampled_parts, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    print('After quick balanced cap:', len(metadata_df))
    display(metadata_df['internal_label'].value_counts().reindex(INTERNAL_LABELS).fillna(0).astype(int).rename('count').to_frame())

metadata_path = DRIVE_META_DIR / f'{EXP_ID.lower()}_metadata_{RUN_MODE}.csv'
metadata_df.to_csv(metadata_path, index=False)
print('metadata:', metadata_path)


In [ ]:
# Cell 5 — Group-protected train/val/test split

def split_score(df, train_subjects, val_subjects, test_subjects):
    global_dist = df['internal_label'].value_counts(normalize=True).reindex(INTERNAL_LABELS).fillna(0.0)
    score = 0.0
    for subjects, target_ratio in [(train_subjects, 0.70), (val_subjects, 0.15), (test_subjects, 0.15)]:
        part = df[df['subject'].isin(subjects)]
        if part.empty:
            return 1e9
        dist = part['internal_label'].value_counts(normalize=True).reindex(INTERNAL_LABELS).fillna(0.0)
        score += float(np.abs(dist - global_dist).sum())
        score += abs(len(part) / len(df) - target_ratio) * 2.0
        missing = (part['internal_label'].value_counts().reindex(INTERNAL_LABELS).fillna(0) == 0).sum()
        score += float(missing) * 0.35
    return score


def choose_subject_split(df, attempts=1000):
    subjects = sorted(df['subject'].unique())
    rng = np.random.default_rng(SEED)
    best = None
    best_score = 1e9
    for _ in range(attempts):
        shuffled = list(subjects)
        rng.shuffle(shuffled)
        n = len(shuffled)
        n_train = max(1, int(round(n * 0.70)))
        n_val = max(1, int(round(n * 0.15)))
        train_subjects = set(shuffled[:n_train])
        val_subjects = set(shuffled[n_train:n_train + n_val])
        test_subjects = set(shuffled[n_train + n_val:])
        if not test_subjects:
            test_subjects = {shuffled[-1]}
            train_subjects.discard(shuffled[-1])
        score = split_score(df, train_subjects, val_subjects, test_subjects)
        if score < best_score:
            best_score = score
            best = train_subjects, val_subjects, test_subjects
    return best, best_score

(train_subjects, val_subjects, test_subjects), best_score = choose_subject_split(metadata_df)
print('Split subjects:', len(train_subjects), len(val_subjects), len(test_subjects), 'score:', round(best_score, 4))

split_by_subject = {}
for s in train_subjects:
    split_by_subject[s] = 'train'
for s in val_subjects:
    split_by_subject[s] = 'val'
for s in test_subjects:
    split_by_subject[s] = 'test'
metadata_df['split'] = metadata_df['subject'].map(split_by_subject)
if metadata_df['split'].isna().any():
    raise RuntimeError('Some rows did not receive split assignment.')

split_path = DRIVE_META_DIR / f'{EXP_ID.lower()}_metadata_{RUN_MODE}_split.csv'
metadata_df.to_csv(split_path, index=False)
print('split metadata:', split_path)

split_counts = metadata_df.groupby(['split', 'internal_label']).size().unstack(fill_value=0).reindex(columns=INTERNAL_LABELS).fillna(0).astype(int)
display(split_counts)
subject_counts = metadata_df.groupby('split')['subject'].nunique().rename('subject_count').to_frame()
display(subject_counts)

# Leakage check.
sets = {split: set(metadata_df[metadata_df['split'] == split]['subject']) for split in ['train', 'val', 'test']}
assert sets['train'].isdisjoint(sets['val'])
assert sets['train'].isdisjoint(sets['test'])
assert sets['val'].isdisjoint(sets['test'])
print('Driver/subject leakage check passed.')


In [ ]:
# Cell 6 — Dataset and dataloaders
class DriverActionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row.image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        target = int(row.target_idx)
        return image, target

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.20, contrast=0.20, saturation=0.15, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_df = metadata_df[metadata_df['split'] == 'train'].reset_index(drop=True)
val_df = metadata_df[metadata_df['split'] == 'val'].reset_index(drop=True)
test_df = metadata_df[metadata_df['split'] == 'test'].reset_index(drop=True)

train_ds = DriverActionDataset(train_df, train_tfms)
val_ds = DriverActionDataset(val_df, val_tfms)
test_ds = DriverActionDataset(test_df, val_tfms)

class_counts = train_df['target_idx'].value_counts().reindex(range(len(INTERNAL_LABELS))).fillna(0).astype(float).values
class_weights = len(train_df) / (len(INTERNAL_LABELS) * np.maximum(class_counts, 1.0))
class_weights = np.clip(class_weights, 0.25, 4.0)
print('class_weights:', {IDX_TO_LABEL[i]: round(float(w), 3) for i, w in enumerate(class_weights)})

sampler = None
shuffle = True
if USE_WEIGHTED_SAMPLER:
    sample_weights = train_df['target_idx'].map({i: class_weights[i] for i in range(len(class_weights))}).astype(float).values
    sampler = WeightedRandomSampler(weights=torch.DoubleTensor(sample_weights), num_samples=len(sample_weights), replacement=True)
    shuffle = False

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=shuffle, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
print('loader sizes:', len(train_ds), len(val_ds), len(test_ds))


In [ ]:
# Cell 7 — Model, training, and evaluation helpers
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)


def build_model(backbone, num_classes):
    if backbone == 'mobilenet_v3_large':
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model
    if backbone == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
        return model
    raise ValueError('Unsupported backbone: ' + backbone)


def set_backbone_trainable(model, trainable):
    for name, param in model.named_parameters():
        # Keep classifier trainable.
        if 'classifier' in name:
            param.requires_grad = True
        else:
            param.requires_grad = bool(trainable)


def evaluate_model(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='eval', leave=False):
            images = images.to(DEVICE)
            targets = targets.to(DEVICE)
            logits = model(images)
            loss = criterion(logits, targets)
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            total_loss += float(loss.item()) * images.size(0)
            all_labels.extend(targets.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())
    labels_np = np.array(all_labels)
    preds_np = np.array(all_preds)
    metrics = {
        'loss': total_loss / max(1, len(loader.dataset)),
        'accuracy': accuracy_score(labels_np, preds_np),
        'macro_f1': f1_score(labels_np, preds_np, average='macro', zero_division=0),
        'weighted_f1': f1_score(labels_np, preds_np, average='weighted', zero_division=0),
    }
    return metrics, labels_np, preds_np, np.array(all_probs)


def train_one_backbone(backbone):
    model = build_model(backbone, len(INTERNAL_LABELS)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    best = {'val_macro_f1': -1.0, 'epoch': -1}
    history = []
    ckpt_path = DRIVE_CKPT_ROOT / f'{EXP_ID}-{backbone}-best.pth'

    for epoch in range(1, EPOCHS + 1):
        set_backbone_trainable(model, trainable=epoch > FREEZE_BACKBONE_EPOCHS)
        model.train()
        running = 0.0
        seen = 0
        for images, targets in tqdm(train_loader, desc=f'{backbone} epoch {epoch}/{EPOCHS}', leave=False):
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits = model(images)
                loss = criterion(logits, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += float(loss.item()) * images.size(0)
            seen += images.size(0)
        val_metrics, _, _, _ = evaluate_model(model, val_loader)
        row = {
            'epoch': epoch,
            'train_loss': running / max(1, seen),
            **{f'val_{k}': v for k, v in val_metrics.items()},
        }
        history.append(row)
        print(backbone, row)
        if val_metrics['macro_f1'] > best['val_macro_f1']:
            best = {'val_macro_f1': float(val_metrics['macro_f1']), 'epoch': epoch, 'path': str(ckpt_path)}
            ckpt = {
                'experiment_id': EXP_ID,
                'experiment_name': EXP_NAME,
                'run_mode': RUN_MODE,
                'backbone': backbone,
                'img_size': IMG_SIZE,
                'labels': INTERNAL_LABELS,
                'label_to_idx': LABEL_TO_IDX,
                'state_dict': model.state_dict(),
                'best_val_macro_f1': float(val_metrics['macro_f1']),
                'epoch': epoch,
                'created_at_utc': datetime.now(timezone.utc).isoformat(),
            }
            torch.save(ckpt, ckpt_path)
            print('Saved best checkpoint:', ckpt_path)
    return {'backbone': backbone, 'best': best, 'history': history, 'checkpoint': str(ckpt_path)}


In [ ]:
# Cell 8 — Train selected backbones
summaries = []
for backbone in BACKBONES:
    summaries.append(train_one_backbone(backbone))
summary_df = pd.DataFrame([
    {
        'backbone': item['backbone'],
        'best_val_macro_f1': item['best']['val_macro_f1'],
        'best_epoch': item['best']['epoch'],
        'checkpoint': item['checkpoint'],
    }
    for item in summaries
]).sort_values('best_val_macro_f1', ascending=False).reset_index(drop=True)
display(summary_df)
summary_path = DRIVE_ARTIFACT_ROOT / f'{EXP_ID.lower()}_backbone_summary_{RUN_MODE}.csv'
summary_df.to_csv(summary_path, index=False)
print('summary:', summary_path)
BEST_BACKBONE = str(summary_df.iloc[0]['backbone'])
BEST_CHECKPOINT = Path(str(summary_df.iloc[0]['checkpoint']))
print('BEST:', BEST_BACKBONE, BEST_CHECKPOINT)


In [ ]:
# Cell 9 — Test evaluation, confusion matrix, and exports
checkpoint = torch.load(BEST_CHECKPOINT, map_location='cpu')
best_model = build_model(checkpoint['backbone'], len(checkpoint['labels'])).to(DEVICE)
best_model.load_state_dict(checkpoint['state_dict'])
best_model.eval()

test_metrics, y_true, y_pred, y_prob = evaluate_model(best_model, test_loader)
print('Test metrics:', test_metrics)

report_dict = classification_report(y_true, y_pred, target_names=INTERNAL_LABELS, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()
report_path = DRIVE_ARTIFACT_ROOT / f'{EXP_ID.lower()}_classification_report_{RUN_MODE}.csv'
report_df.to_csv(report_path)
display(report_df)

cm = confusion_matrix(y_true, y_pred, labels=list(range(len(INTERNAL_LABELS))))
cm_path = DRIVE_ARTIFACT_ROOT / f'{EXP_ID.lower()}_confusion_matrix_{RUN_MODE}.csv'
pd.DataFrame(cm, index=INTERNAL_LABELS, columns=INTERNAL_LABELS).to_csv(cm_path)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(INTERNAL_LABELS)))
ax.set_yticks(range(len(INTERNAL_LABELS)))
ax.set_xticklabels(INTERNAL_LABELS, rotation=45, ha='right')
ax.set_yticklabels(INTERNAL_LABELS)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'{EXP_ID} confusion matrix ({RUN_MODE})')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax)
fig.tight_layout()
cm_png = DRIVE_ARTIFACT_ROOT / f'{EXP_ID.lower()}_confusion_matrix_{RUN_MODE}.png'
fig.savefig(cm_png, dpi=160)
plt.show()
print('confusion matrix:', cm_path, cm_png)

# Save label map and test predictions.
label_map_path = DRIVE_CKPT_ROOT / f'{EXP_ID}-label-map.json'
label_map_path.write_text(json.dumps({'labels': INTERNAL_LABELS, 'label_to_idx': LABEL_TO_IDX}, ensure_ascii=False, indent=2), encoding='utf-8')

test_pred_df = test_df.copy()
test_pred_df['pred_idx'] = y_pred
test_pred_df['pred_label'] = [IDX_TO_LABEL[int(i)] for i in y_pred]
test_pred_df['correct'] = test_pred_df['target_idx'].values == y_pred
for idx, label in IDX_TO_LABEL.items():
    test_pred_df[f'prob_{label}'] = y_prob[:, idx]
pred_path = DRIVE_ARTIFACT_ROOT / f'{EXP_ID.lower()}_test_predictions_{RUN_MODE}.csv'
test_pred_df.to_csv(pred_path, index=False)
print('label map:', label_map_path)
print('predictions:', pred_path)


In [ ]:
# Cell 10 — Action gate policy table and promotion summary
FTR_PROMOTION_POLICY = {
    'telefonla_konusma': {
        'promotes_to_ftr': True,
        'needs_support': 'temporal persistence + optional phone/hand-ear evidence',
        'note': 'State Farm phone-call classes c2/c4 only. Texting c1/c3 is kept separate as phone_use_non_call.',
    },
    'su_icme': {
        'promotes_to_ftr': True,
        'needs_support': 'temporal persistence + optional bottle/cup evidence',
        'note': 'State Farm c6 is direct drinking positive.',
    },
    'arkaya_bakma_candidate': {
        'promotes_to_ftr': False,
        'needs_support': 'head/torso orientation gate before final arkaya_bakma',
        'note': 'State Farm c7 means reaching behind; it is not strictly looking back.',
    },
    'phone_use_non_call': {
        'promotes_to_ftr': False,
        'needs_support': 'support/hard-negative only',
        'note': 'Texting is not phone-call; do not write telefonla_konusma directly.',
    },
    'passenger_interaction_candidate': {
        'promotes_to_ftr': False,
        'needs_support': 'gaze/head-pose and passenger context',
        'note': 'Talking to passenger is not enough for etrafa_bakinma.',
    },
    'other_distraction_hard_negative': {
        'promotes_to_ftr': False,
        'needs_support': 'hard-negative',
        'note': 'Radio/hair-makeup used to reduce false positives.',
    },
    'safe_or_no_event': {
        'promotes_to_ftr': False,
        'needs_support': 'none',
        'note': 'No event.',
    },
}
policy_path = DRIVE_ARTIFACT_ROOT / f'{EXP_ID.lower()}_promotion_policy.json'
policy_path.write_text(json.dumps(FTR_PROMOTION_POLICY, ensure_ascii=False, indent=2), encoding='utf-8')
print(policy_path)
display(pd.DataFrame(FTR_PROMOTION_POLICY).transpose())


In [ ]:
# Cell 11 — Sample predictions and evidence preview grid
sample_df = test_pred_df.sample(n=min(24, len(test_pred_df)), random_state=SEED).reset_index(drop=True)
cols = 4
rows = math.ceil(len(sample_df) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.4))
axes = np.array(axes).reshape(-1)
for ax, row in zip(axes, sample_df.itertuples(index=False)):
    img = Image.open(row.image_path).convert('RGB')
    ax.imshow(img)
    color = 'green' if row.correct else 'red'
    ax.set_title(f'T:{row.internal_label}\nP:{row.pred_label}', color=color, fontsize=9)
    ax.axis('off')
for ax in axes[len(sample_df):]:
    ax.axis('off')
fig.tight_layout()
preview_path = DRIVE_ARTIFACT_ROOT / f'{EXP_ID.lower()}_sample_predictions_{RUN_MODE}.png'
fig.savefig(preview_path, dpi=160)
plt.show()
print('preview:', preview_path)


In [ ]:
# Cell 12 — Optional local road-video gate note
print('Existing 3 Teknofest road-facing videos are NOT driver-action validation data.')
print('Expected runtime behavior: DRIVER-EXP-001 gate must run first; if not cabin/action-evaluable, DACT-EXP-020B should not produce sofor_eylemi.')
print('This notebook evaluates on State Farm subject-disjoint split only. External validation with AUC/DMD remains pending unless added later.')


In [ ]:
# Cell 13 — Save summary JSON and markdown report
created_at = datetime.now(timezone.utc).isoformat()
per_class = report_df.loc[INTERNAL_LABELS][['precision', 'recall', 'f1-score', 'support']].reset_index().rename(columns={'index': 'label'})
summary = {
    'experiment_id': EXP_ID,
    'experiment_name': EXP_NAME,
    'run_mode': RUN_MODE,
    'created_at_utc': created_at,
    'source_dataset': 'State Farm Distracted Driver Detection',
    'backbones': BACKBONES,
    'best_backbone': BEST_BACKBONE,
    'best_checkpoint': str(BEST_CHECKPOINT),
    'label_map_path': str(label_map_path),
    'metadata_path': str(split_path),
    'test_metrics': {k: float(v) for k, v in test_metrics.items()},
    'per_class': per_class.to_dict(orient='records'),
    'promotion_policy_path': str(policy_path),
    'classification_report_csv': str(report_path),
    'confusion_matrix_csv': str(cm_path),
    'confusion_matrix_png': str(cm_png),
    'sample_predictions_png': str(preview_path),
    'notes': [
        'Single-frame classifier baseline; final event should use temporal voting.',
        'Texting is kept separate from telefonla_konusma.',
        'arkaya_bakma_candidate is weak reaching-behind evidence, not final looking-back.',
        'esneme, sigara_icme, emniyet_kemeri_ihlali and etrafa_bakinma require separate specialist phases.',
    ],
}
summary_json_path = DRIVE_ARTIFACT_ROOT / f'{EXP_ID.lower()}_summary_{RUN_MODE}.json'
summary_json_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

rows = []
for item in per_class.to_dict(orient='records'):
    rows.append(
        f"| `{item['label']}` | {item['precision']:.4f} | {item['recall']:.4f} | {item['f1-score']:.4f} | {int(item['support'])} |"
    )

report_lines = [
    '# DACT-EXP-020B Driver Action Classifier',
    '',
    f'Tarih: {created_at}',
    '',
    '## Amaç',
    '',
    "State Farm Distracted Driver Detection verisiyle ilk sürücü eylemi classifier baseline'ını kurmak.",
    '',
    '## Konfigürasyon',
    '',
    f'* Run mode: `{RUN_MODE}`',
    f'* Best backbone: `{BEST_BACKBONE}`',
    f'* Best checkpoint: `{BEST_CHECKPOINT}`',
    f'* Metadata: `{split_path}`',
    '',
    '## Test Metrikleri',
    '',
    f"* Accuracy: `{test_metrics['accuracy']:.4f}`",
    f"* Macro-F1: `{test_metrics['macro_f1']:.4f}`",
    f"* Weighted-F1: `{test_metrics['weighted_f1']:.4f}`",
    '',
    '## Per-Class Metrikler',
    '',
    '| Label | Precision | Recall | F1 | Support |',
    '|---|---:|---:|---:|---:|',
    *rows,
    '',
    '## Karar Notu',
    '',
    '* `telefonla_konusma` yalnız State Farm c2/c4 phone-call sınıflarından gelir; texting sınıfları ayrı `phone_use_non_call` tutulur.',
    '* `su_icme` doğrudan drinking sınıfından gelir.',
    '* `arkaya_bakma_candidate`, reaching-behind weak label olarak kalır; head/torso gate olmadan final etiket değildir.',
    "* Bu model tek-frame classifier baseline'ıdır; runtime event için temporal voting ve `DRIVER-EXP-001` gate gerekir.",
    '* `sigara_icme`, `esneme`, `emniyet_kemeri_ihlali`, `etrafa_bakinma` ayrı specialist fazları olarak devam eder.',
    '',
    '## Çıktılar',
    '',
    f'* Summary JSON: `{summary_json_path}`',
    f'* Classification report: `{report_path}`',
    f'* Confusion matrix: `{cm_png}`',
    f'* Sample predictions: `{preview_path}`',
]
DRIVE_REPORT_PATH.write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
print('summary:', summary_json_path)
print('report:', DRIVE_REPORT_PATH)
